# Étape 2 — Interface commune (issue #5, pour le 30 septembre)

**Tout est prêt à exécuter.** Exécute dans l'ordre avec Maj+Entrée et lis les explications. Runtime : **T4 GPU** (seulement pour la cellule 6).

## Pourquoi une « interface commune » ?

Lounis écrit le DE **séquentiel** (CPU), toi le DE **GPU**. À la fin, on compare les deux. Pour que la comparaison soit honnête, il faut qu'ils optimisent **exactement les mêmes fonctions**, sur **les mêmes domaines**, avec **le même décalage** `o`, **le même budget**, et qu'ils écrivent leurs résultats **dans le même format**. Ce contrat, on l'écrit **une seule fois**, dans des fichiers partagés :

| Fichier | Rôle |
|---|---|
| `src/common/benchmarks.h` | les 4 fonctions corrigées + domaines + décalage + budget |
| `src/common/results_csv.h` | écrire une ligne de résultat dans un CSV |
| `src/common/test_benchmarks.cpp` | les tests CPU |
| `src/common/test_gpu.cu` | le test GPU |
| `docs/INTERFACE.md` | la documentation du contrat (pour Lounis, Lisa et l'article) |

## Ce que cette interface corrige par rapport au code du prof

| Bug du PSO | Correction ici |
|---|---|
| 2 : Griewank `produit = 0` | `p = 1.0` |
| 4 : pas de décalage (o = 0) | vrai `o` aléatoire, fixe et reproductible |
| 5 : domaine [−5.12, 5.12] pour tout | domaines CEC 2005 par fonction |
| 9 : biais + `float` → précision 3·10⁻⁵ | `double`, et on renvoie directement l'**erreur** (0 à l'optimum) |
| deux copies de la fonction (CPU et GPU) | **une seule** copie `__host__ __device__` |

### Cellule 1 — Récupérer le dépôt et créer ta branche

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
token   = userdata.get("GITHUB_TOKEN")
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO} || (cd /content/{REPO} && git checkout -q main && git pull -q --rebase)
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout -q main && git pull -q --rebase
!git checkout -B amina/interface-commune
!mkdir -p src/common docs
!git branch --show-current

### Cellule 2 — `benchmarks.h` : le cœur du contrat

À lire en priorité :
- **`#define HD __host__ __device__`** : sous `nvcc`, chaque fonction marquée `HD` est compilée **deux fois**, une pour le CPU et une pour le GPU. Fini les deux copies qui divergent (voir la question sur `host_fitness_function`).
- **`evaluate(f, x, o, D)`** : le seul point d'entrée. Le DE ne connaît que cette fonction.
- **`make_shift(f, D)`** : le décalage `o`. On utilise un petit générateur déterministe (*splitmix64*) avec une graine fixe `1000·f + D` : n'importe qui, sur n'importe quelle machine, obtient le **même** `o`. C'est ce qui rend les résultats reproductibles.
- **`max_fe(D)`** : le budget du sujet, 10⁴ × D évaluations.

In [ ]:
%%writefile /content/DE_CUDA/src/common/benchmarks.h
// benchmarks.h -- INTERFACE COMMUNE du projet DE_CUDA (issue #5)
// Utilise par la version sequentielle (src/seq) ET par la version GPU (src/cuda).
// Ne pas modifier sans prevenir les deux autres membres du groupe.
#pragma once
#include <cmath>
#include <cstdint>
#include <cstring>
#include <vector>

// Une meme fonction compilee pour le CPU et pour le GPU :
// sous nvcc (__CUDACC__ defini) -> __host__ __device__ ; sous g++ -> rien.
#ifdef __CUDACC__
#define HD __host__ __device__
#else
#define HD
#endif

// ---------------------------------------------------------------- identifiants
enum FuncId { F_SPHERE = 0, F_ROSENBROCK = 1, F_GRIEWANK = 2, F_RASTRIGIN = 3 };
static const int NUM_FUNCS = 4;
#define DE_PI 3.14159265358979323846   // macro : utilisable sur CPU et GPU

inline const char* func_name(int f) {
    static const char* names[] = {"sphere", "rosenbrock", "griewank", "rastrigin"};
    return (f >= 0 && f < NUM_FUNCS) ? names[f] : "inconnue";
}
// "sphere" -> 0, ... ; -1 si inconnu
inline int func_id(const char* name) {
    for (int f = 0; f < NUM_FUNCS; f++) if (std::strcmp(name, func_name(f)) == 0) return f;
    return -1;
}

// ---------------------------------------------------------------- domaines (CEC 2005)
HD inline double lower_bound(int f) {
    switch (f) {
        case F_GRIEWANK:  return -600.0;
        case F_RASTRIGIN: return -5.0;
        default:          return -100.0;   // sphere, rosenbrock
    }
}
HD inline double upper_bound(int f) { return -lower_bound(f); }

// ---------------------------------------------------------------- les 4 fonctions
// Toutes renvoient l'ERREUR f(x) - f(x*) : 0 exactement a l'optimum x = o.
// Le biais CEC (-450, 390, -180, -330) n'est PAS ajoute (voir bug 9 :
// en float, pres de 450, on ne peut pas mesurer une erreur < 3e-5).

// Sphere : somme z_i^2, z = x - o
HD inline double sphere(const double* x, const double* o, int D) {
    double s = 0.0;
    for (int i = 0; i < D; i++) { double z = x[i] - o[i]; s += z * z; }
    return s;
}
// Rosenbrock : somme_{i<D-1} 100 (z_i^2 - z_{i+1})^2 + (z_i - 1)^2, z = x - o + 1
HD inline double rosenbrock(const double* x, const double* o, int D) {
    double s = 0.0;
    for (int i = 0; i < D - 1; i++) {
        double zi  = x[i]     - o[i]     + 1.0;
        double zi1 = x[i + 1] - o[i + 1] + 1.0;
        double a = zi * zi - zi1, b = zi - 1.0;
        s += 100.0 * a * a + b * b;
    }
    return s;
}
// Griewank : somme z_i^2 / 4000 - produit cos(z_i / sqrt(i+1)) + 1, z = x - o
HD inline double griewank(const double* x, const double* o, int D) {
    double s = 0.0, p = 1.0;                       // p = 1 (bug 2 corrige)
    for (int i = 0; i < D; i++) {
        double z = x[i] - o[i];
        s += z * z / 4000.0;
        p *= cos(z / sqrt((double)(i + 1)));
    }
    return s - p + 1.0;
}
// Rastrigin : somme z_i^2 - 10 cos(2 pi z_i) + 10, z = x - o
HD inline double rastrigin(const double* x, const double* o, int D) {
    double s = 0.0;
    for (int i = 0; i < D; i++) {
        double z = x[i] - o[i];
        s += z * z - 10.0 * cos(2.0 * DE_PI * z) + 10.0;
    }
    return s;
}
// Point d'entree unique : c'est CETTE fonction que le DE appelle.
HD inline double evaluate(int f, const double* x, const double* o, int D) {
    switch (f) {
        case F_SPHERE:     return sphere(x, o, D);
        case F_ROSENBROCK: return rosenbrock(x, o, D);
        case F_GRIEWANK:   return griewank(x, o, D);
        default:           return rastrigin(x, o, D);
    }
}

// ---------------------------------------------------------------- decalage o
// Generateur deterministe (splitmix64) : meme graine -> memes nombres,
// sur n'importe quelle machine. CPU et GPU utilisent donc EXACTEMENT le meme o.
inline uint64_t splitmix64(uint64_t& s) {
    uint64_t z = (s += 0x9E3779B97F4A7C15ULL);
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9ULL;
    z = (z ^ (z >> 27)) * 0x94D049BB133111EBULL;
    return z ^ (z >> 31);
}
// o_i tire uniformement dans 80 % du domaine, graine = 1000*f + D (fixee).
inline std::vector<double> make_shift(int f, int D) {
    uint64_t s = 1000ULL * (uint64_t)f + (uint64_t)D;
    double lo = 0.8 * lower_bound(f), hi = 0.8 * upper_bound(f);
    std::vector<double> o(D);
    for (int i = 0; i < D; i++) {
        double u = (double)(splitmix64(s) >> 11) * (1.0 / 9007199254740992.0); // [0,1)
        o[i] = lo + (hi - lo) * u;
    }
    return o;
}

// ---------------------------------------------------------------- budget
inline long long max_fe(int D) { return 10000LL * D; }   // 10^4 x D (sujet)


### Cellule 3 — `results_csv.h` : le format des résultats

Chaque run écrit **une ligne** : `algo,func,D,N,seed,best_error,time_s,FE`. Le séquentiel, le GPU v1 et le GPU v2 écriront dans ce même format : Lisa pourra tout analyser avec un seul script.

In [ ]:
%%writefile /content/DE_CUDA/src/common/results_csv.h
// results_csv.h -- FORMAT COMMUN des resultats (issue #5)
// Une ligne par run :  algo,func,D,N,seed,best_error,time_s,FE
#pragma once
#include <cstdio>

static const char* CSV_HEADER = "algo,func,D,N,seed,best_error,time_s,FE";

// Ajoute une ligne au fichier `path` (cree l'en-tete si le fichier est neuf).
inline bool append_result(const char* path, const char* algo, const char* func,
                          int D, int N, unsigned seed, double best_error,
                          double time_s, long long fe) {
    FILE* t = std::fopen(path, "r");
    bool fresh = (t == nullptr);
    if (t) std::fclose(t);
    FILE* fp = std::fopen(path, "a");
    if (!fp) return false;
    if (fresh) std::fprintf(fp, "%s\n", CSV_HEADER);
    std::fprintf(fp, "%s,%s,%d,%d,%u,%.6e,%.6f,%lld\n",
                 algo, func, D, N, seed, best_error, time_s, fe);
    std::fclose(fp);
    return true;
}


### Cellule 4 — Les tests CPU

Le critère de validation de l'issue #5 : **f(o) = 0 pour les 4 fonctions et D = 10, 50, 100**. On vérifie aussi que `o` reste dans le domaine, on affiche des valeurs de référence, et on teste l'écriture CSV.

In [ ]:
%%writefile /content/DE_CUDA/src/common/test_benchmarks.cpp
// test_benchmarks.cpp -- valide l'interface commune (critere de l'issue #5)
#include <cstdio>
#include "benchmarks.h"
#include "results_csv.h"

int main() {
    int dims[3] = {10, 50, 100};
    int ok = 1;
    printf("Test 1 : f(o) = 0 exactement\n");
    for (int f = 0; f < NUM_FUNCS; f++)
        for (int d : dims) {
            std::vector<double> o = make_shift(f, d);
            double v = evaluate(f, o.data(), o.data(), d);
            bool good = (v == 0.0);
            ok &= good;
            printf("  %-10s D=%3d  f(o) = %.3e  %s\n", func_name(f), d, v, good ? "OK" : "ECHEC");
        }
    printf("Test 2 : le decalage reste dans 80 %% du domaine\n");
    for (int f = 0; f < NUM_FUNCS; f++) {
        std::vector<double> o = make_shift(f, 100);
        double mn = o[0], mx = o[0];
        for (double v : o) { if (v < mn) mn = v; if (v > mx) mx = v; }
        bool good = mn >= 0.8 * lower_bound(f) && mx <= 0.8 * upper_bound(f);
        ok &= good;
        printf("  %-10s domaine [%g, %g]  o dans [%.2f, %.2f]  %s\n", func_name(f),
               lower_bound(f), upper_bound(f), mn, mx, good ? "OK" : "ECHEC");
    }
    printf("Test 3 : valeurs de reference (a comparer avec NumPy)\n");
    for (int f = 0; f < NUM_FUNCS; f++) {
        std::vector<double> o = make_shift(f, 10), x(10);
        for (int i = 0; i < 10; i++) x[i] = o[i] + 0.1 * (i + 1);   // point connu
        printf("  REF %s %.17g\n", func_name(f), evaluate(f, x.data(), o.data(), 10));
    }
    printf("Test 4 : ecriture CSV\n");
    std::remove("test_resultats.csv");
    append_result("test_resultats.csv", "test", "sphere", 10, 50, 1, 1.234e-9, 0.5, 100000);
    append_result("test_resultats.csv", "test", "rastrigin", 10, 50, 2, 3.5, 0.6, 100000);
    FILE* fp = fopen("test_resultats.csv", "r"); char line[256];
    while (fgets(line, sizeof line, fp)) printf("  %s", line);
    fclose(fp);
    printf("\n%s\n", ok ? "=== TOUS LES TESTS PASSENT ===" : "=== ECHEC ===");
    return ok ? 0 : 1;
}


In [ ]:
%cd /content/DE_CUDA/src/common
!g++ -std=c++14 -O2 -Wall -o test_benchmarks test_benchmarks.cpp && ./test_benchmarks

Tu dois lire **`=== TOUS LES TESTS PASSENT ===`** à la fin.

### Cellule 5 — Contre-vérification indépendante avec NumPy

Un test qui compare le code à lui-même ne prouve rien. On recalcule les 4 valeurs de référence (test 3) **avec NumPy**, écrit indépendamment, directement à partir des formules mathématiques. Au point testé, `z_i = x_i − o_i = 0.1·(i+1)`.

In [ ]:
import numpy as np, subprocess, re
out = subprocess.run(["./test_benchmarks"], capture_output=True, text=True, cwd="/content/DE_CUDA/src/common").stdout
ref = {m[0]: float(m[1]) for m in re.findall(r"REF (\w+) ([\d.eE+-]+)", out)}
z = 0.1 * np.arange(1, 11)
numpy_vals = {
    "sphere":     np.sum(z**2),
    "rosenbrock": np.sum(100*((z[:-1]+1)**2 - (z[1:]+1))**2 + z[:-1]**2),
    "griewank":   np.sum(z**2)/4000 - np.prod(np.cos(z/np.sqrt(np.arange(1, 11)))) + 1,
    "rastrigin":  np.sum(z**2 - 10*np.cos(2*np.pi*z) + 10),
}
for f, v in numpy_vals.items():
    ecart = abs(ref[f] - v) / max(1, abs(v))
    print(f"{f:10s} C++ = {ref[f]:.15g}   NumPy = {v:.15g}   ecart relatif = {ecart:.1e}   {'OK' if ecart < 1e-12 else 'ECHEC'}")

### Cellule 6 — Le test GPU (nécessite le runtime T4)

Même calcul, mais la fonction `evaluate` tourne **sur le GPU** dans un kernel. Si CPU et GPU donnent le même résultat (à 10⁻¹² près), c'est que `__host__ __device__` fonctionne et qu'il n'y a plus qu'**une seule** version de chaque fonction.

Remarque la macro **`CUDA_CHECK`** : elle vérifie chaque appel CUDA et affiche un message clair en cas de problème. On la réutilisera dans tout le DE (tâche 3.1).

In [ ]:
%%writefile /content/DE_CUDA/src/common/test_gpu.cu
// test_gpu.cu -- verifie que les fonctions donnent le MEME resultat sur GPU et sur CPU
#include <cstdio>
#include <cmath>
#include "benchmarks.h"

#define CUDA_CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \
    printf("Erreur CUDA %s ligne %d : %s\n", __FILE__, __LINE__, cudaGetErrorString(e)); return 1; } } while (0)

// Un seul thread suffit ici : on teste la fonction, pas la performance.
__global__ void k_eval(int f, const double* x, const double* o, int D, double* out) {
    if (blockIdx.x == 0 && threadIdx.x == 0) *out = evaluate(f, x, o, D);
}

int main() {
    const int D = 100;
    int ok = 1;
    double *dx, *dout, *dshift;
    CUDA_CHECK(cudaMalloc(&dx, D * sizeof(double)));
    CUDA_CHECK(cudaMalloc(&dshift, D * sizeof(double)));
    CUDA_CHECK(cudaMalloc(&dout, sizeof(double)));
    for (int f = 0; f < NUM_FUNCS; f++) {
        std::vector<double> o = make_shift(f, D), x(D);
        for (int i = 0; i < D; i++) x[i] = o[i] + 0.01 * (i + 1);
        double cpu = evaluate(f, x.data(), o.data(), D);
        CUDA_CHECK(cudaMemcpy(dx, x.data(), D * sizeof(double), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(dshift, o.data(), D * sizeof(double), cudaMemcpyHostToDevice));
        k_eval<<<1, 1>>>(f, dx, dshift, D, dout);
        CUDA_CHECK(cudaGetLastError());
        double gpu;
        CUDA_CHECK(cudaMemcpy(&gpu, dout, sizeof(double), cudaMemcpyDeviceToHost));
        double rel = fabs(gpu - cpu) / fmax(1.0, fabs(cpu));
        bool good = rel < 1e-12;
        ok &= good;
        printf("%-10s CPU = %.15e  GPU = %.15e  ecart rel = %.1e  %s\n",
               func_name(f), cpu, gpu, rel, good ? "OK" : "ECHEC");
    }
    cudaFree(dx); cudaFree(dshift); cudaFree(dout);
    printf("\n%s\n", ok ? "=== CPU et GPU donnent le meme resultat ===" : "=== ECHEC ===");
    return ok ? 0 : 1;
}


In [ ]:
%cd /content/DE_CUDA/src/common
!nvcc -O2 -arch=sm_75 -o test_gpu test_gpu.cu && ./test_gpu

Tu dois lire **`=== CPU et GPU donnent le meme resultat ===`**.

### Cellule 7 — La documentation du contrat

Pour Lounis, Lisa, et pour l'article (section 4.2 *Benchmark functions*).

In [ ]:
%%writefile /content/DE_CUDA/docs/INTERFACE.md
# Interface commune DE_CUDA (issue #5)

Contrat partagé entre la version séquentielle (`src/seq`) et la version GPU (`src/cuda`).
**Ne pas modifier sans prévenir les deux autres membres du groupe.**

## Fichiers

| Fichier | Contenu |
|---|---|
| `src/common/benchmarks.h` | 4 fonctions de test, domaines, décalage `o`, budget |
| `src/common/results_csv.h` | écriture d'une ligne de résultat dans un CSV |
| `src/common/test_benchmarks.cpp` | tests CPU (f(o) = 0, domaines, valeurs de référence, CSV) |
| `src/common/test_gpu.cu` | test GPU : même résultat que le CPU |

## Fonctions (en `double`, renvoient l'erreur f(x) − f(x*), 0 à l'optimum)

| id | nom | domaine | formule (z = x − o) |
|---|---|---|---|
| 0 | sphere | [−100, 100] | Σ z_i² |
| 1 | rosenbrock | [−100, 100] | Σ_{i<D−1} 100 (z_i² − z_{i+1})² + (z_i − 1)², avec z = x − o + 1 |
| 2 | griewank | [−600, 600] | Σ z_i²/4000 − Π cos(z_i/√(i+1)) + 1 |
| 3 | rastrigin | [−5, 5] | Σ z_i² − 10 cos(2π z_i) + 10 |

- Appel unique : `evaluate(f, x, o, D)`, marqué `__host__ __device__` (utilisable sur CPU et GPU).
- Biais CEC 2005 **non ajouté** : on travaille directement sur l'erreur (précision, bug 9).
- Décalage : `make_shift(f, D)` tire `o` dans 80 % du domaine avec un générateur déterministe
  (splitmix64, graine 1000·f + D). CPU et GPU utilisent donc exactement le même `o`.
- Budget : `max_fe(D) = 10^4 × D` évaluations, **en comptant les N évaluations de l'initialisation**.

## Ligne de commande (commune aux deux versions)

```
./programme <fonction> <D> <N> <seed> [fichier.csv]
./sde sphere 10 50 1 results/sde.csv
```

## Format CSV (une ligne par run)

```
algo,func,D,N,seed,best_error,time_s,FE
sde,sphere,10,50,1,1.234000e-09,0.500000,100000
```

| colonne | sens |
|---|---|
| algo | `sde`, `cuda_v1`, `cuda_v2`, … |
| func | nom de la fonction |
| D, N, seed | dimension, population, graine du run |
| best_error | meilleure erreur trouvée (notation scientifique) |
| time_s | temps de calcul en secondes (sans l'initialisation du contexte CUDA) |
| FE | nombre d'évaluations réellement faites (≤ 10^4 × D) |


### Cellule 8 — Sauvegarder sur ta branche

On n'ajoute **que** les sources et la doc, pas les exécutables ni le CSV de test.

In [ ]:
%cd /content/DE_CUDA
!git add src/common/benchmarks.h src/common/results_csv.h src/common/test_benchmarks.cpp src/common/test_gpu.cu docs/INTERFACE.md
!git status --short
!git commit -m "Interface commune : benchmarks.h, format CSV, tests CPU et GPU (refs #5)"
!git push -u origin amina/interface-commune

### Dernière étape — la pull request

1. Sur github.com/amenacer/DE_CUDA → bandeau jaune **Compare & pull request**.
2. Titre : `Interface commune (benchmarks.h, CSV)`.
3. Description : colle la sortie des cellules 4, 5 et 6 (les lignes OK), puis une ligne `closes #5`.
4. **Reviewers** : Lounis **et** Lisa — c'est leur contrat aussi, ils doivent le lire et l'approuver.
5. **Create pull request**, puis carte **#5** → **In Review**.

⚠️ **Ne fusionne pas toi-même.** Lounis doit relire, poser ses questions, et c'est lui qui fusionne. Une fois fusionné, plus personne ne modifie ces fichiers sans prévenir les autres.